<h1 align="center">🇨🇭 Swiss GTFS and NeTEx Analysis 2026</h1>

# 🇨🇭 Swiss GTFS and NeTEx Analysis 2026

This notebook explores and compares two official Swiss public transport datasets 
for the timetable year 2026: the **GTFS** feed and the **NeTEx** feed, both 
published by the Open Data Platform Mobility Switzerland.

The goal is to identify the **Minimum Viable Denominator (MVD)**, the smallest 
common set of fields and concepts that both formats can reliably provide, at 
three levels:

- **Station level** — do both formats describe the same stops?
- **Line level** — do both formats describe the same public lines?
- **Calendar level** — do both formats cover the same timetable period?

This follows the same methodology developed in the Austrian analysis and serves 
as one country case in a broader multi-country comparison.

## Table of Contents

**Part 1 — Switzerland GTFS Exploration**
1. Load the Switzerland GTFS ZIP and inspect file sizes
2. Define a helper function to read GTFS tables from the ZIP
3. Load all GTFS tables: stops, routes, trips, agency, calendar, 
   calendar_dates, frequencies, transfers
4. Explore each table: shape, columns, missing values
5. First summary: unique routes, service IDs, timetable date range
6. Check the stop hierarchy: location_type and parent_station

**Part 2 — Switzerland NeTEx Exploration**
7. Load the Switzerland NeTEx ZIP and list all XML files
8. Inspect the largest XML file and identify the frame structure
9. Group XML files by type: TIMETABLE, SITE, SERVICECALENDAR, SERVICE, COMMON, RESOURCE
10. Check one example file per group to confirm frame content

**Part 3 — Switzerland NeTEx Extraction**
11. Extract stop data from the SITE file (StopPlace and Quay)
12. Extract line data from the SERVICE file
13. Extract calendar validity window from the SERVICECALENDAR file

**Part 4 — Prepare datasets for comparison**
14. Define the GTFS station subset and the NeTEx StopPlace subset
15. Compare data quality across both subsets
16. Clean and match station IDs between GTFS and NeTEx
17. Summarize station-level match rate
18. Compare line-level fields and unique public line labels
19. Normalize line labels and compute overlap between GTFS and NeTEx

**Part 5 — Conclusions**
20. Station-level conclusion: match rate and unmatched records
21. Line-level conclusion: public label overlap
22. Calendar-level note: shared validity window

## Data Source

This analysis uses two official datasets published by the 
Open Data Platform Mobility Switzerland (opentransportdata.swiss), 
operated by the Swiss Federal Office of Transport (BAV):

🇨🇭**GTFS Dataset**  
Timetable 2026 — GTFS Static  
https://data.opentransportdata.swiss/dataset/timetable-2026-gtfs2020/resource/1d8312b8-7e27-4cc6-b6ac-3207fa92b5b4

🇨🇭**NeTEx Dataset**  
Timetable 2026 — NeTEx  
https://data.opentransportdata.swiss/dataset/timetablenetex_2026/resource/cce2bb3c-2221-4f39-ab12-891e6b8dbf1e

Both datasets were accessed on 8 April 2026 under the license:  
*Non-commercial Allowed / Commercial Allowed / Reference Required*

# Part 1: Switzerland GTFS Exploration

## Load the Switzerland GTFS ZIP

I start with the Switzerland GTFS ZIP file.  
I use the same helper-function approach as in the Austria notebook, so that the loading logic stays consistent across countries.

All third-party and standard-library imports used in this notebook are consolidated here.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import re
import xml.etree.ElementTree as ET

In [2]:
zip_path = Path("data/switzerland_gtfs_2026-04-01.zip")
print(zip_path)
print("Exists:", zip_path.exists())

data/switzerland_gtfs_2026-04-01.zip
Exists: True


## Helper function to read GTFS tables from the ZIP

This helper function reads one GTFS text file directly from the ZIP.  
It keeps the workflow simple and avoids unpacking everything manually.

In [3]:
def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If caller passed exact path, use it
        if filename in names:
            target = filename
        else:
            # Otherwise find it anywhere in the zip
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]
            if not matches:
                raise FileNotFoundError(f"{filename} not found. Example: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches for {filename}: {matches}")
            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

## Load the main GTFS files

In [4]:
# Check files sizes
with zipfile.ZipFile(zip_path, "r") as z:
    for info in z.infolist():
        print(f"{info.filename:40s} {info.file_size / 1024**2:.1f} MB")

agency.txt                               0.0 MB
routes.txt                               0.2 MB
trips.txt                                171.6 MB
stops.txt                                9.7 MB
calendar.txt                             2.7 MB
calendar_dates.txt                       176.5 MB
transfers.txt                            19.0 MB
feed_info.txt                            0.0 MB
frequencies.txt                          0.1 MB
stop_times.txt                           1532.4 MB


## Observation

The Switzerland GTFS feed is much larger than the Austria feed, especially for trips and calendar_dates. (The feed also includes a very large stop_times.txt, but that table is not needed for the station-, line-, and calendar-level MVD comparisons below, so it is not loaded here.)
So I keep the same general loading logic as in the Austria notebook, but I load the large tables more carefully and only keep the columns that are relevant for the thesis.

## Explore each table

In [5]:
stops = read_gtfs_from_zip(zip_path, "stops.txt")
print(f"stops: {stops.shape[0]:,} rows x {stops.shape[1]} cols")
print(f"Missing: {stops.isna().sum()[stops.isna().sum() > 0].to_dict()}")
stops.head()

stops: 97,067 rows x 8 cols
Missing: {'location_type': 61800, 'parent_station': 35267, 'platform_code': 85046, 'original_stop_id': 2570}


,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station,platform_code,original_stop_id
0,1100008,"Zell (Wiesental), Wilder Mann",47.710084,7.859648,NaN,Parent1100008,NaN,ch:1:sloid:1100008
1,1100009,"Zell (Wiesental), Grönland",47.713191,7.862909,NaN,Parent1100009,NaN,ch:1:sloid:1100009
2,1100010,Atzenbach,47.714618,7.872350,NaN,Parent1100010,NaN,ch:1:sloid:1100010
3,1100011,"Mambach, Brücke",47.728209,7.877470,NaN,Parent1100011,NaN,ch:1:sloid:1100011
4,1100012,"Mambach, Mühlschau",47.734082,7.881387,NaN,Parent1100012,NaN,ch:1:sloid:1100012


In [6]:
routes = read_gtfs_from_zip(zip_path, "routes.txt")
print(f"routes: {routes.shape[0]:,} rows x {routes.shape[1]} cols")
print(f"Missing: {routes.isna().sum()[routes.isna().sum() > 0].to_dict()}")
routes.head()

routes: 4,883 rows x 6 cols
Missing: {'route_long_name': 4883}


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type
0,91-10-A-j26-1,78,S10,NaN,S,109
1,91-10-B-j26-1,11,S10,NaN,S,109
2,91-10-C-j26-1,65,S10,NaN,S,109
3,91-10-D-j26-1,11,10,NaN,EXT,117
4,91-10-E-j26-1,3849,10,NaN,T,900


In [7]:
agency = read_gtfs_from_zip(zip_path, "agency.txt")
print("loaded agency")
print(agency.shape)
print(agency.columns.tolist())
agency.head()

loaded agency
(465, 6)
['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone']


,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone
0,87_LEX,Société Nationale des Chemins de fer Français,http://www.sbb.ch/,Europe/Berlin,DE,0848 44 66 88
1,11,Schweizerische Bundesbahnen SBB,http://www.sbb.ch/,Europe/Berlin,DE,0848 44 66 88
2,823,Basler Verkehrsbetriebe,http://www.sbb.ch/,Europe/Berlin,DE,0848 44 66 88
3,37,Baselland Transport,http://www.sbb.ch/,Europe/Berlin,DE,0848 44 66 88
4,827,Städtische Verkehrsbetriebe Bern,http://www.sbb.ch/,Europe/Berlin,DE,0848 44 66 88


In [8]:
calendar = read_gtfs_from_zip(zip_path, "calendar.txt")   
print(f"calendar: {calendar.shape[0]:,} rows x {calendar.shape[1]} cols")  
print(f"Missing: {calendar.isna().sum()[calendar.isna().sum() > 0].to_dict()}")  
calendar.head()  

calendar: 45,276 rows x 10 cols
Missing: {}


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,TA,1,1,1,1,1,1,1,20251214,20261212
1,TA+00100,0,0,0,0,1,0,1,20251214,20261212
2,TA+00210,1,1,1,1,0,0,0,20251214,20261212
3,TA+00300,0,0,0,0,0,1,1,20251214,20261212
4,TA+00520,1,1,1,1,1,1,1,20251214,20261212


In [9]:
calendar_dates = read_gtfs_from_zip(zip_path, "calendar_dates.txt")
print(f"calendar_dates: {calendar_dates.shape[0]:,} rows x {calendar_dates.shape[1]} cols")
print(f"Missing: {calendar_dates.isna().sum()[calendar_dates.isna().sum() > 0].to_dict()}")
calendar_dates.head()

calendar_dates: 6,869,292 rows x 3 cols
Missing: {}


,service_id,date,exception_type
0,TA+00100,20260628,2
1,TA+00100,20260703,2
2,TA+00100,20260705,2
3,TA+00100,20260710,2
4,TA+00100,20260712,2


In [10]:
frequencies = read_gtfs_from_zip(zip_path, "frequencies.txt")
print(f"frequencies: {frequencies.shape[0]:,} rows x {frequencies.shape[1]} cols")
print(f"Missing: {frequencies.isna().sum()[frequencies.isna().sum() > 0].to_dict()}")
frequencies.head()

frequencies: 1,816 rows x 5 cols
Missing: {}


,trip_id,start_time,end_time,headway_secs,exact_times
0,1.TA.93-244-0-j26-1.1.H,08:00:00,16:30:00,60,0
1,2.TA.93-244-0-j26-1.1.H,08:00:00,16:00:00,60,0
2,3.TA.93-244-0-j26-1.1.H,08:00:00,16:15:00,60,0
3,4.TA.93-244-0-j26-1.1.H,09:00:00,15:45:00,60,0
4,5.TA.93-244-0-j26-1.1.H,08:00:00,15:45:00,60,0


In [11]:
transfers = read_gtfs_from_zip(zip_path, "transfers.txt")
print(f"transfers: {transfers.shape[0]:,} rows x {transfers.shape[1]} cols")
print(f"Missing: {transfers.isna().sum()[transfers.isna().sum() > 0].to_dict()}")
transfers.head()

transfers: 218,904 rows x 8 cols
Missing: {'from_route_id': 93019, 'to_route_id': 93019, 'from_trip_id': 93019, 'to_trip_id': 93019, 'min_transfer_time': 125885}


,from_stop_id,to_stop_id,from_route_id,to_route_id,from_trip_id,to_trip_id,transfer_type,min_transfer_time
0,1100008,1100008,NaN,NaN,NaN,NaN,2,300.0
1,1100009,1100009,NaN,NaN,NaN,NaN,2,300.0
2,1100010,1100010,NaN,NaN,NaN,NaN,2,300.0
3,1100011,1100011,NaN,NaN,NaN,NaN,2,300.0
4,1100012,1100012,NaN,NaN,NaN,NaN,2,300.0


In [12]:
trips = read_gtfs_from_zip(zip_path, "trips.txt")
print(f"trips: {trips.shape[0]:,} rows x {trips.shape[1]} cols")
print(f"Missing: {trips.isna().sum()[trips.isna().sum() > 0].to_dict()}")
trips.head()

trips: 1,275,824 rows x 9 cols
Missing: {'block_id': 1088255, 'original_trip_id': 133512, 'hints': 71693}


,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,original_trip_id,hints
0,91-10-A-j26-1,TA+lbs00,1.TA.91-10-A-j26-1.1.H,Zürich HB,12888,0,NaN,ch:1:sjyid:100058:12888-002,2 NF VN
1,91-10-A-j26-1,TA+lbs00,10.TA.91-10-A-j26-1.1.H,Zürich HB,12790,0,NaN,ch:1:sjyid:100058:12790-001,2 NF VN
2,91-10-A-j26-1,TA+tal00,100.TA.91-10-A-j26-1.5.H,Zürich Selnau,12880,0,NaN,ch:1:sjyid:100058:12880-003,2 NF
3,91-10-A-j26-1,TA+ral00,101.TA.91-10-A-j26-1.5.H,Zürich Selnau,12968,0,NaN,ch:1:sjyid:100058:12968-002,2 NF
4,91-10-A-j26-1,TA+tal00,102.TA.91-10-A-j26-1.5.H,Zürich Selnau,12844,0,NaN,ch:1:sjyid:100058:12844-003,2 NF


## First summary of the Switzerland GTFS dataset

In this step, I make a first general summary of the Switzerland GTFS feed. I check how large the main tables are, how many unique routes and service IDs exist, and which date range is covered by the timetable data.

The goal is to understand the overall structure of the dataset before moving on to a deeper comparison with NeTEx.

In [13]:
print("stops:", len(stops))
print("routes:", len(routes))
print("trips:", len(trips))
print("calendar:", len(calendar))
print("calendar_dates:", len(calendar_dates))
print("agency:", len(agency))
print("frequencies:", len(frequencies))
print("transfers:", len(transfers))

print("\nunique route_id in routes:", routes["route_id"].nunique())
print("unique route_id in trips:", trips["route_id"].nunique())
print("unique service_id in calendar:", calendar["service_id"].nunique())
print("unique service_id in calendar_dates:", calendar_dates["service_id"].nunique())

print("\ncalendar start_date min:", calendar["start_date"].min())
print("calendar end_date max:", calendar["end_date"].max())
print("calendar_dates date min:", calendar_dates["date"].min())
print("calendar_dates date max:", calendar_dates["date"].max())

stops: 97067
routes: 4883
trips: 1275824
calendar: 45276
calendar_dates: 6869292
agency: 465
frequencies: 1816
transfers: 218904

unique route_id in routes: 4883
unique route_id in trips: 4883
unique service_id in calendar: 45276
unique service_id in calendar_dates: 45260

calendar start_date min: 20251214
calendar end_date max: 20261212
calendar_dates date min: 20251214
calendar_dates date max: 20261212


## Observation from the first GTFS summary

The Switzerland GTFS dataset is much larger than the Austria dataset, especially in the trips and calendar_dates tables. The feed contains *4,883 unique routes* and more than *45,000 service IDs*. The timetable validity period runs from *2025-12-14 to 2026-12-12* in both calendar sources.

This shows that the dataset is internally consistent and covers one full timetable period, but it is clearly much larger and more complex than the Austria case.

## Check of the stop hierarchy

In this step, I look at the stop hierarchy in the Switzerland GTFS dataset. I check the distribution of the `location_type` field and also how often `parent_station` is used.

This helps me understand whether the feed mainly contains platform-level stops, station-level stops, or a mixture of both.

In [14]:
print("location_type value counts:")
print(stops["location_type"].value_counts(dropna=False))

print("\nparent_station non-null count:")
print(stops["parent_station"].notna().sum())

location_type value counts:
location_type
NaN    61800
1.0    35267
Name: count, dtype: int64

parent_station non-null count:
61800


## Observation from the stop hierarchy check

The Switzerland GTFS stops table mainly contains two groups: records with missing `location_type` and records with `location_type = 1`. In addition, parent_station is used very often.

This suggests that the feed contains many child stops linked to parent stations, so the stop structure is hierarchical. This is important for the station-level comparison later, because I will need to think carefully about which GTFS stop objects should be treated as the closest match to NeTEx station-level objects.

# Part 2: Switzerland NeTEx Exploration

## Inspect Switzerland NeTEx files

In this part of the notebook, I now start the first inspection of the Switzerland NeTEx dataset.

I follow the same general approach as in the Austria notebook. First, I load the raw NeTEx ZIP file and inspect its internal structure. Then I check which XML files are included and whether the dataset contains the main elements I need for the comparison with GTFS.

The goal of this step is to see whether I can identify the main comparable objects for the MVD, especially:
- stop or station-related objects
- line-related objects
- calendar or validity-related objects

In [15]:
netex_zip = Path("data/switzerland_netex_2026-03-31.zip")

print(netex_zip)
print("Exists:", netex_zip.exists())

data/switzerland_netex_2026-03-31.zip
Exists: True


In [16]:
with zipfile.ZipFile(netex_zip, "r") as z:
    infos = z.infolist()

df_files = pd.DataFrame([{
    "name": i.filename,
    "size_mb": round(i.file_size / (1024 * 1024), 2)
} for i in infos])

display(df_files.sort_values("size_mb", ascending=False).head(20))

xml_files = df_files[df_files["name"].str.lower().str.endswith(".xml")].sort_values("size_mb", ascending=False)

print("XML files:", len(xml_files))
display(xml_files.head(20))

,name,size_mb
177,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,654.98
201,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,437.51
200,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,400.07
206,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,372.01
174,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,216.76
188,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,190.62
162,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,170.41
187,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,161.17
253,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,158.28
176,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,156.84


XML files: 272


,name,size_mb
177,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,654.98
201,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,437.51
200,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,400.07
206,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,372.01
174,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,216.76
188,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,190.62
162,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,170.41
187,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,161.17
253,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,158.28
176,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TI...,156.84


## Observation

> At first glance, Switzerland does not have more XML files than Austria, but the dataset looks much heavier and is probably more consolidated. Austria was split into many smaller XML files, while Switzerland packages much more content into fewer, much larger files.

### Next Steps

- Pick one very large Swiss XML file
- Open only the first few KB
- Inspect the header and frame structure

This helps understand whether the file is structured similarly to the Austrian 
NeTEx files and what kind of content it contains.

In [17]:
# Pick the largest XML file and print the first part

with zipfile.ZipFile(netex_zip, "r") as z:
    xml_name = xml_files.iloc[0]["name"]
    print("Opening:", xml_name)

    with z.open(xml_name) as f:
        head = f.read(5000)   # first ~5 KB
        print(head.decode("utf-8", errors="replace"))

Opening: PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_173_267_202603280103.xml
﻿<?xml version="1.0" encoding="utf-8"?>
<PublicationDelivery xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:siri="http://www.siri.org.uk/siri" xsi:schemaLocation="http://www.netex.org.uk/netex http://netex.uk/netex/schema/1.10/xsd/NeTEx_publication.xsd" version="1.10" xmlns="http://www.netex.org.uk/netex">
  <!--Profile: SBBSplitted-->
  <PublicationTimestamp>2026-03-28T02:32:35.8736548+01:00</PublicationTimestamp>
  <ParticipantRef>MENTZ</ParticipantRef>
  <Description>NeTEx Export, Version: 19.78.0.6748</Description>
  <dataObjects>
    <CompositeFrame id="ch:1:CompositeFrame:j26" version="any">
      <ValidBetween>
        <FromDate>2025-12-14T00:00:00</FromDate>
        <ToDate>2026-12-12T23:59:59</ToDate>
      </ValidBetween>
      <FrameDefaults>
        <DefaultLocale>
          <TimeZoneOffset>1</TimeZoneOffset>
          <SummerTimeZo

## First observation from the XML header

Here I check whether the file contains a PublicationDelivery structure, a validity period, and references to the main NeTEx frames.

This gives me a first clue about whether the Swiss file contains stop, line, timetable, or calendar-related information.

In [18]:
header_text = head.decode("utf-8", errors="replace")

keywords = [
    "PublicationDelivery",
    "CompositeFrame",
    "ValidBetween",
    "ResourceFrame",
    "SiteFrame",
    "ServiceFrame",
    "ServiceCalendarFrame",
    "TimetableFrame",
    "StopPlace",
    "Quay",
    "ScheduledStopPoint",
    "Line",
    "ServiceJourney"
]

for k in keywords:
    print(f"{k}: {k in header_text}")

PublicationDelivery: True
CompositeFrame: True
ValidBetween: True
ResourceFrame: False
SiteFrame: False
ServiceFrame: False
ServiceCalendarFrame: False
TimetableFrame: True
StopPlace: False
Quay: False
ScheduledStopPoint: True
Line: True
ServiceJourney: True


## Comment on the first Switzerland NeTEx inspection

The first inspection shows that the Swiss NeTEx export is structured in large 
XML files with different functions. The file opened here is clearly 
timetable-oriented, as it contains a `TimetableFrame` with elements such as 
`ServiceJourney`, `ScheduledStopPoint`, `Call`, and `LineRef`.

This suggests that the Swiss NeTEx export is split into function-specific file 
groups rather than storing all information in one mixed XML file. The next step 
is therefore to identify which files relate to timetable data, stop/site data, 
service data, and calendar data before starting the extraction.

This interpretation is consistent with the Swiss NeTEx Cookbook, which explains 
that the export is split into several frame-based files such as `TimetableFrame`, 
`ServiceFrame`, `SiteFrame`, and `ServiceCalendarFrame`.

Source: [Swiss NeTEx Cookbook, Open Data Platform Mobility Switzerland](https://opentransportdata.swiss/en/cookbook/timetable-cookbook/netex/)

## Check the main Swiss NeTEx file types

In this step, I inspect a few of the largest XML files from the Switzerland NeTEx ZIP.

The goal is to see which main frame each file contains and whether the same validity period appears across files. This helps me understand which files are most relevant for later extraction of stops, lines, and calendar-related data.

In [19]:
def inspect_netex_header(zip_path, xml_name, n_bytes=120000):
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_name) as f:
            txt = f.read(n_bytes).decode("utf-8", errors="replace")

    frame_match = re.search(
        r"<(TimetableFrame|ServiceFrame|SiteFrame|ServiceCalendarFrame|ResourceFrame|CommonFrame)\b",
        txt
    )
    from_match = re.search(r"<FromDate>(.*?)</FromDate>", txt)
    to_match = re.search(r"<ToDate>(.*?)</ToDate>", txt)

    return {
        "xml_name": xml_name,
        "frame_type": frame_match.group(1) if frame_match else None,
        "from_date": from_match.group(1) if from_match else None,
        "to_date": to_match.group(1) if to_match else None
    }

sample_xml = xml_files["name"].head(10).tolist()

header_rows = []
for xml_name in sample_xml:
    header_rows.append(inspect_netex_header(netex_zip, xml_name))

swiss_netex_headers = pd.DataFrame(header_rows)
swiss_netex_headers

pd.set_option("display.max_colwidth", None)
swiss_netex_headers

,xml_name,frame_type,from_date,to_date
0,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_173_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
1,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_197_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
2,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_196_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
3,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_202_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
4,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_170_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
5,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_184_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
6,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_158_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
7,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_183_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
8,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_249_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
9,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_172_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59


## Output

The first ten large XML files all appear to be `TimetableFrame` files, and they all share the same validity period from 2025-12-14 to 2026-12-12.

This suggests that the Swiss NeTEx export contains multiple large timetable files with one common timetable window. The next step is therefore to look for other file groups, especially files related to site, service, and calendar information.

## Look for non-timetable NeTEx files

The first large files I checked are all timetable files.  
So now I look for other XML files that may correspond to site, service, or calendar information.

The goal is to find the files that are most relevant for later extraction of:
- stop or station-related data
- line-related data
- calendar or validity-related data

### Group the Switzerland NeTEx files by file name

In this step, I do a simple check of the XML file names inside the Switzerland NeTEx ZIP.

I search for keywords in the file names, such as:
- `TIMETABLE`
- `SITE`
- `SERVICECALENDAR`
- `SERVICE`
- `COMMON`
- `RESOURCE`

This helps me see how the Swiss NeTEx export is organised before I start extracting data from the XML files.

In [20]:
file_group_keywords = ["TIMETABLE", "SITE", "SERVICECALENDAR", "SERVICE", "COMMON", "RESOURCE"]

for kw in file_group_keywords:
    matches = [n for n in xml_files["name"] if f"_{kw}_" in n.upper()]
    print(f"\n{kw}: {len(matches)}")
    for name in matches[:10]:
        print(name)


TIMETABLE: 267
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_173_267_202603280103.xml
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_197_267_202603280103.xml
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_196_267_202603280103.xml
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_202_267_202603280103.xml
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_170_267_202603280103.xml
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_184_267_202603280103.xml
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_158_267_202603280103.xml
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_183_267_202603280103.xml
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_249_267_202603280103.xml
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_172_267_202603280103.xml

SITE: 1
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SITE_1_1_202603280103.xml

SERVICECALENDAR: 1
PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SERVICECALENDAR_1_1_202603280103.xml

SERVICE: 1
PR

## Observation

The output shows that most XML files belong to the `TIMETABLE` group. In addition, there is one `SITE` file, one `SERVICECALENDAR` file, one `COMMON` file, one `RESOURCE` file, and one main `SERVICE` file.

This suggests that the Swiss NeTEx export is strongly split by function. The timetable information is distributed across many XML files, while other parts of the data, such as site, service, and calendar information, seem to be stored in only a few dedicated files.

Because of this, the next step is to inspect these smaller non-timetable files and confirm which one should be used for stops, lines, and calendar extraction.

## Check one example file from each Swiss NeTEx group

Here, I take one example XML file from each file group that I identified before (`TIMETABLE`, `SITE`, `SERVICECALENDAR`, `SERVICE`, `COMMON`, `RESOURCE`) and inspect only the beginning of the file.

The goal is to check two things:
1. which main frame type the file actually contains
2. whether the validity period is the same across the different file groups

In [21]:
def inspect_one_candidate(zip_path, names, label, n_bytes=120000):
    if not names:
        print(f"\n{label}: no filename match found")
        return None

    xml_name = names[0]
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_name) as f:
            txt = f.read(n_bytes).decode("utf-8", errors="replace")

    frame_match = re.search(
        r"<(TimetableFrame|ServiceFrame|SiteFrame|ServiceCalendarFrame|ResourceFrame|CommonFrame)\b",
        txt
    )
    from_match = re.search(r"<FromDate>(.*?)</FromDate>", txt)
    to_match = re.search(r"<ToDate>(.*?)</ToDate>", txt)

    result = {
        "group": label,
        "xml_name": xml_name,
        "frame_type": frame_match.group(1) if frame_match else None,
        "from_date": from_match.group(1) if from_match else None,
        "to_date": to_match.group(1) if to_match else None
    }
    return result

candidates = {}
for kw in file_group_keywords:
    candidates[kw] = [n for n in xml_files["name"] if kw in n.upper()]

rows = []
for kw in file_group_keywords:
    result = inspect_one_candidate(netex_zip, candidates[kw], kw)
    if result is not None:
        rows.append(result)

candidate_headers = pd.DataFrame(rows)
candidate_headers

,group,xml_name,frame_type,from_date,to_date
0,TIMETABLE,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_TIMETABLE_173_267_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
1,SITE,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SITE_1_1_202603280103.xml,SiteFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
2,SERVICECALENDAR,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SERVICECALENDAR_1_1_202603280103.xml,ServiceCalendarFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
3,SERVICE,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SERVICE_1_1_202603280103.xml,ServiceFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
4,COMMON,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_COMMON_1_1_202603280103.xml,TimetableFrame,2025-12-14T00:00:00,2026-12-12T23:59:59
5,RESOURCE,PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_RESOURCE_1_1_202603280103.xml,ResourceFrame,2025-12-14T00:00:00,2026-12-12T23:59:59


## Observation from the file group check

The output shows that the Swiss NeTEx export is clearly split into different functional file groups.

The example files confirm the following:
- the `TIMETABLE` file contains a `TimetableFrame`
- the `SITE` file contains a `SiteFrame`
- the `SERVICECALENDAR` file contains a `ServiceCalendarFrame`
- the `SERVICE` file contains a `ServiceFrame`
- the `RESOURCE` file contains a `ResourceFrame`

All checked files also share the same validity period from 2025-12-14 to 2026-12-12.

One interesting detail is that the file with `COMMON` in its name was identified here as a `TimetableFrame`, not as a `CommonFrame`. This means that the file names are helpful, but they are not always enough on their own. It is better to confirm the actual frame type from the XML content.

This result is useful for the next step, because it shows which Swiss NeTEx files should be used later for:
- stop or station extraction (`SITE`)
- line extraction (`SERVICE`)
- calendar extraction (`SERVICECALENDAR`)

## Extract stop data from the Swiss NeTEx SITE file


### Select the Swiss NeTEx SITE file

In this cell, I define which XML file I want to use for the first NeTEx extraction.

I choose the `SITE` file because the previous checks showed that this file contains a `SiteFrame`. This is the file that should contain stop or station-related information.

The goal is to use this file for the first extraction of Swiss NeTEx stops.

In [22]:
NETEX_NS = "{http://www.netex.org.uk/netex}"

site_file = "PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SITE_1_1_202603280103.xml"
print(site_file)

PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SITE_1_1_202603280103.xml


### Create a function to extract stop data from the SITE file

In this cell, I define a function that reads the Swiss NeTEx `SITE` file and extracts stop-related objects.

What this function does:
- opens the XML file from the ZIP
- reads it step by step in a memory-friendly way
- looks for `Quay` and `StopPlace`
- extracts:
  - the stop ID
  - the stop name
  - the latitude
  - the longitude
- adds a label to show whether the object is a `Quay` or a `StopPlace`

This gives me a first stop table from the Swiss NeTEx data that I can later compare with GTFS.

In [23]:
def extract_netex_stops_from_site(netex_zip_path, xml_name):
    rows = []

    with zipfile.ZipFile(netex_zip_path, "r") as z:
        with z.open(xml_name) as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                
                if elem.tag == NETEX_NS + "Quay":
                    stop_id = elem.get("id")
                    name_el = elem.find(NETEX_NS + "Name")
                    stop_name = name_el.text.strip() if (name_el is not None and name_el.text) else None

                    lon_el = elem.find(NETEX_NS + "Centroid/" + NETEX_NS + "Location/" + NETEX_NS + "Longitude")
                    lat_el = elem.find(NETEX_NS + "Centroid/" + NETEX_NS + "Location/" + NETEX_NS + "Latitude")

                    stop_lon = float(lon_el.text) if (lon_el is not None and lon_el.text) else None
                    stop_lat = float(lat_el.text) if (lat_el is not None and lat_el.text) else None

                    rows.append({
                        "stop_id": stop_id,
                        "stop_name": stop_name,
                        "stop_lat": stop_lat,
                        "stop_lon": stop_lon,
                        "stop_type": "Quay"
                    })
                    elem.clear()

                elif elem.tag == NETEX_NS + "StopPlace":
                    stop_id = elem.get("id")
                    name_el = elem.find(NETEX_NS + "Name")
                    stop_name = name_el.text.strip() if (name_el is not None and name_el.text) else None

                    lon_el = elem.find(NETEX_NS + "Centroid/" + NETEX_NS + "Location/" + NETEX_NS + "Longitude")
                    lat_el = elem.find(NETEX_NS + "Centroid/" + NETEX_NS + "Location/" + NETEX_NS + "Latitude")

                    stop_lon = float(lon_el.text) if (lon_el is not None and lon_el.text) else None
                    stop_lat = float(lat_el.text) if (lat_el is not None and lat_el.text) else None

                    rows.append({
                        "stop_id": stop_id,
                        "stop_name": stop_name,
                        "stop_lat": stop_lat,
                        "stop_lon": stop_lon,
                        "stop_type": "StopPlace"
                    })
                    elem.clear()

    return pd.DataFrame(rows).drop_duplicates(subset=["stop_id"])

In [24]:
netex_stops_ch = extract_netex_stops_from_site(netex_zip, site_file)

print(netex_stops_ch.shape)
print(netex_stops_ch.isna().mean().sort_values(ascending=False))
netex_stops_ch.head(10)

(77177, 5)
stop_name    0.542299
stop_id      0.000000
stop_lat     0.000000
stop_lon     0.000000
stop_type    0.000000
dtype: float64


,stop_id,stop_name,stop_lat,stop_lon,stop_type
0,ch:2:Quay:10:10000,None,47.014775,6.766901,Quay
1,ch:2:Quay:10:10001,None,47.014634,6.766659,Quay
2,ch:2:StopPlace:10,"Petit-Martel-Est, halte",47.014750,6.766874,StopPlace
3,ch:2:StopPlace:1100008,"Zell (Wiesental), Wilder Mann",47.710084,7.859648,StopPlace
4,ch:2:StopPlace:1100009,"Zell (Wiesental), Grönland",47.713191,7.862909,StopPlace
5,ch:2:StopPlace:1100010,Atzenbach,47.714618,7.872350,StopPlace
6,ch:2:StopPlace:1100011,"Mambach, Brücke",47.728209,7.877470,StopPlace
7,ch:2:StopPlace:1100012,"Mambach, Mühlschau",47.734082,7.881387,StopPlace
8,ch:2:StopPlace:1100013,"Mambach, Silbersau",47.739519,7.882232,StopPlace
9,ch:2:StopPlace:1100014,"Fröhnd (Schwarzw), Wühre",47.754366,7.889131,StopPlace


## Observation

The extraction from the Swiss NeTEx `SITE` file worked and produced a large stop table with **77,177 rows**.

Main observations:
- the stop table contains both `Quay` and `StopPlace` objects
- `stop_id`, `stop_lat`, `stop_lon`, and `stop_type` have no missing values
- `stop_name` has a high missing share (about 54%)

The first rows suggest that many unnamed records are `Quay` objects, while `StopPlace` objects more often contain readable stop names. This means the Swiss NeTEx `SITE` file provides usable stop-related data, but the choice of object type will matter for the later comparison with GTFS.

For the station-level comparison, it may be more practical to focus mainly on `StopPlace` objects, or at least to separate `StopPlace` and `Quay` clearly before matching.

## Check the stop types in the Swiss NeTEx stop table

In this step, I look more closely at the two stop types in the extracted Swiss NeTEx stop table: `StopPlace` and `Quay`.

The goal is to understand:
- how many rows belong to each stop type
- whether missing names are more common in one type than the other
- which stop type is more suitable for the later comparison with GTFS

In [25]:
print("Stop type counts:")
print(netex_stops_ch["stop_type"].value_counts(dropna=False))

print("\nMissing stop_name by stop type:")
print(
    netex_stops_ch
    .groupby("stop_type")["stop_name"]
    .apply(lambda s: s.isna().mean())
    .sort_values(ascending=False)
)

Stop type counts:
stop_type
Quay         41853
StopPlace    35324
Name: count, dtype: int64

Missing stop_name by stop type:
stop_type
Quay         1.0
StopPlace    0.0
Name: stop_name, dtype: float64


## Observation

The most important observation is that the missing names come only from the `Quay` objects:
- all `Quay` rows have a missing `stop_name`
- all `StopPlace` rows have a non-missing `stop_name`

So in other words:
- the Swiss NeTEx stop data contains many `Quay` and many `StopPlace` objects
- but only the `StopPlace` objects are useful if I want readable stop names
- this means that `StopPlace` is likely the better starting point for the later comparison with GTFS

In [26]:
stopplace_ch = netex_stops_ch[netex_stops_ch["stop_type"] == "StopPlace"].copy()
quay_ch = netex_stops_ch[netex_stops_ch["stop_type"] == "Quay"].copy()

print("StopPlace shape:", stopplace_ch.shape)
print("Quay shape:", quay_ch.shape)

print("\nStopPlace missing stop_name %:", stopplace_ch["stop_name"].isna().mean())
print("Quay missing stop_name %:", quay_ch["stop_name"].isna().mean())

print("\nExample StopPlace rows:")
display(stopplace_ch.head(10))

print("\nExample Quay rows:")
display(quay_ch.head(10))

StopPlace shape: (35324, 5)
Quay shape: (41853, 5)

StopPlace missing stop_name %: 0.0
Quay missing stop_name %: 1.0

Example StopPlace rows:


,stop_id,stop_name,stop_lat,stop_lon,stop_type
2,ch:2:StopPlace:10,"Petit-Martel-Est, halte",47.014750,6.766874,StopPlace
3,ch:2:StopPlace:1100008,"Zell (Wiesental), Wilder Mann",47.710084,7.859648,StopPlace
4,ch:2:StopPlace:1100009,"Zell (Wiesental), Grönland",47.713191,7.862909,StopPlace
5,ch:2:StopPlace:1100010,Atzenbach,47.714618,7.872350,StopPlace
6,ch:2:StopPlace:1100011,"Mambach, Brücke",47.728209,7.877470,StopPlace
7,ch:2:StopPlace:1100012,"Mambach, Mühlschau",47.734082,7.881387,StopPlace
8,ch:2:StopPlace:1100013,"Mambach, Silbersau",47.739519,7.882232,StopPlace
9,ch:2:StopPlace:1100014,"Fröhnd (Schwarzw), Wühre",47.754366,7.889131,StopPlace
10,ch:2:StopPlace:1100015,"Fröhnd (Schwarzw), Unterkastel",47.760593,7.885537,StopPlace
11,ch:2:StopPlace:1100016,Wembach (Baden),47.772832,7.887720,StopPlace



Example Quay rows:


,stop_id,stop_name,stop_lat,stop_lon,stop_type
0,ch:2:Quay:10:10000,None,47.014775,6.766901,Quay
1,ch:2:Quay:10:10001,None,47.014634,6.766659,Quay
567,ch:2:Quay:1100863:8,None,47.652756,9.473561,Quay
569,ch:2:Quay:1100864:2,None,47.649761,9.497968,Quay
571,ch:2:Quay:1100866:1,None,47.594620,9.598777,Quay
572,ch:2:Quay:1100866:2,None,47.594620,9.598777,Quay
574,ch:2:Quay:1100871:1,None,47.826554,10.016925,Quay
575,ch:2:Quay:1100871:3,None,47.826554,10.016925,Quay
576,ch:2:Quay:1100871:4,None,47.826554,10.016925,Quay
578,ch:2:Quay:1100873:1,None,47.545145,9.681072,Quay


## Output

This output confirms the difference between the two stop types.

What I observe:
- the `StopPlace` rows have readable stop names and coordinates
- the `Quay` rows also have coordinates, but the names are missing
- the `StopPlace` examples look like normal station or stop names
- the `Quay` examples look more like technical or platform-level objects

So in simple words:
- `StopPlace` looks much more useful for a direct comparison with GTFS stops
- `Quay` may still be useful later as extra detail, but not as the main object for name-based matching
- because of this, the Swiss NeTEx station-level comparison should probably start with `StopPlace`

## Extract line data from the Swiss NeTEx SERVICE file

In this step, I move from stop extraction to line extraction.

I use the Swiss NeTEx `SERVICE` file, because the earlier file check showed that this file contains a `ServiceFrame`. This is the part of the export where line-related objects are expected.


In [27]:
# Select the file

service_file = "PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SERVICE_1_1_202603280103.xml"
print(service_file)

PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SERVICE_1_1_202603280103.xml


### Create a function to extract lines from the Swiss NeTEx SERVICE file

The function:
- looks for `Line`
- extracts:
  - the line ID
  - the line name
  - the short name, if available
  - the public code, if available

In [28]:
def extract_netex_lines_from_service(netex_zip_path, xml_name):
    rows = []

    with zipfile.ZipFile(netex_zip_path, "r") as z:
        with z.open(xml_name) as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                if elem.tag == NETEX_NS + "Line":
                    route_id = elem.get("id")

                    name_el = elem.find(NETEX_NS + "Name")
                    short_el = elem.find(NETEX_NS + "ShortName")
                    public_el = elem.find(NETEX_NS + "PublicCode")

                    route_name = None
                    if name_el is not None and name_el.text and name_el.text.strip():
                        route_name = name_el.text.strip()
                    elif short_el is not None and short_el.text and short_el.text.strip():
                        route_name = short_el.text.strip()

                    short_name = short_el.text.strip() if (short_el is not None and short_el.text) else None
                    public_code = public_el.text.strip() if (public_el is not None and public_el.text) else None

                    rows.append({
                        "route_id": route_id,
                        "route_name": route_name,
                        "short_name": short_name,
                        "public_code": public_code
                    })

                    elem.clear()

    return pd.DataFrame(rows).drop_duplicates(subset=["route_id"])

In [29]:
netex_lines_ch = extract_netex_lines_from_service(netex_zip, service_file)

print(netex_lines_ch.shape)
print(netex_lines_ch.isna().mean().sort_values(ascending=False))
netex_lines_ch.head(10)

(4014, 4)
route_id       0.0
route_name     0.0
short_name     0.0
public_code    0.0
dtype: float64


,route_id,route_name,short_name,public_code
0,ch:2:Line:06____.B.7,7,7,7
1,ch:2:Line:06____.RB.27,27,27,27
2,ch:2:Line:06____.RE.RE7,RE7,7,RE7
3,ch:2:Line:101.B.23,23,23,23
4,ch:2:Line:101.FUN.23,23,23,23
5,ch:2:Line:103.FUN.22,22,22,22
6,ch:2:Line:104.CC.7,7,7,7
7,ch:2:Line:105.FUN.2840,2840,284,2840
8,ch:2:Line:107.ASC.ASC,ASC,4V,ASC
9,ch:2:Line:107.FUN.FUN,FUN,84,FUN


## Output

The extraction from the Swiss NeTEx `SERVICE` file worked and produced a line table with 4,014 rows and 4 columns.

Main observations:
- the extracted table contains line IDs, line names, short names, and public codes
- no missing values were found in these four selected fields
- the first rows show that the Swiss NeTEx line data contains usable public line labels such as `7`, `27`, `RE7`, `23`, `2840`, `ASC`, and `FUN`

The general difference in structure between GTFS and NeTEx was already observed in the Austria dataset. The Swiss result confirms again that the NeTEx line data provides usable public line information, even though the fields are not organised in exactly the same way as in GTFS.

For the later comparison, this means that I can continue with the same general logic and focus on the most comparable public line labels.

## Extract calendar data from the Swiss NeTEx SERVICECALENDAR file

In this step, I move from line extraction to calendar extraction.

I use the Swiss NeTEx `SERVICECALENDAR` file, because the earlier file check showed that this file contains a `ServiceCalendarFrame`. This is the part of the export that is most relevant for service validity and calendar-related information.

In [30]:
# Select the file

servicecalendar_file = "PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SERVICECALENDAR_1_1_202603280103.xml"
print(servicecalendar_file)

PROD_NETEX_TT_1.10_CHE_SKI_2026_OEV-SCHWEIZ_SERVICECALENDAR_1_1_202603280103.xml


### Create a function to extract the main validity period

In this cell, I define a small function that opens the Swiss NeTEx `SERVICECALENDAR` file and looks for the main validity period.

The function reads only the beginning of the file and extracts:
- `FromDate`
- `ToDate`

This gives me a first simple calendar table from the Swiss NeTEx data.

In [31]:
def extract_netex_calendar_window(netex_zip_path, xml_name, n_bytes=200000):
    with zipfile.ZipFile(netex_zip_path, "r") as z:
        with z.open(xml_name) as f:
            txt = f.read(n_bytes).decode("utf-8", errors="replace")

    from_match = re.search(r"<FromDate>(.*?)</FromDate>", txt)
    to_match = re.search(r"<ToDate>(.*?)</ToDate>", txt)

    from_date = from_match.group(1) if from_match else None
    to_date = to_match.group(1) if to_match else None

    # convert to YYYYMMDD like GTFS
    from_date_clean = from_date[:10].replace("-", "") if from_date else None
    to_date_clean = to_date[:10].replace("-", "") if to_date else None

    calendar_netex_ch = pd.DataFrame([{
        "service_id": "NETEX_CH_2026",
        "start_date": from_date_clean,
        "end_date": to_date_clean
    }])

    return calendar_netex_ch

In [32]:
netex_calendar_ch = extract_netex_calendar_window(netex_zip, servicecalendar_file)

print(netex_calendar_ch.shape)
netex_calendar_ch

(1, 3)


,service_id,start_date,end_date
0,NETEX_CH_2026,20251214,20261212


## Output
The extraction worked and returned one validity window: **2025-12-14 to 2026-12-12**.
This matches what we would expect for a full Swiss timetable year and can later be compared with the GTFS calendar.

# Part 3: Prepare the Switzerland datasets for comparison

In the previous parts, I explored the GTFS and NeTEx data separately and extracted the main objects needed for the comparison.

I now start preparing the Swiss datasets for the actual comparison between GTFS and NeTEx.

## 3.1 Station-level comparison

In this step, I prepare the stop subsets that are most relevant for the station-level comparison.

On the NeTEx side, I keep only `StopPlace`, because the earlier extraction showed that this is the cleaner and more useful stop type for direct comparison.

On the GTFS side, I create the same two station-like subsets that I also checked in the Austria dataset:
- records without `parent_station`
- records with `location_type == 1.0`

This helps me see which GTFS subset is closer to the NeTEx `StopPlace` level.

In [33]:
# NeTEx station subset: keep only StopPlace
netex_stopplace_ch = netex_stops_ch[netex_stops_ch["stop_type"] == "StopPlace"].copy()

print("Swiss NeTEx all stops:", netex_stops_ch.shape)
print("Swiss NeTEx StopPlace only:", netex_stopplace_ch.shape)

netex_stopplace_ch.head()

Swiss NeTEx all stops: (77177, 5)
Swiss NeTEx StopPlace only: (35324, 5)


,stop_id,stop_name,stop_lat,stop_lon,stop_type
2,ch:2:StopPlace:10,"Petit-Martel-Est, halte",47.014750,6.766874,StopPlace
3,ch:2:StopPlace:1100008,"Zell (Wiesental), Wilder Mann",47.710084,7.859648,StopPlace
4,ch:2:StopPlace:1100009,"Zell (Wiesental), Grönland",47.713191,7.862909,StopPlace
5,ch:2:StopPlace:1100010,Atzenbach,47.714618,7.872350,StopPlace
6,ch:2:StopPlace:1100011,"Mambach, Brücke",47.728209,7.877470,StopPlace


In [34]:
# GTFS station-like subsets
gtfs_no_parent_ch = stops[stops["parent_station"].isna()].copy()
gtfs_loc1_ch = stops[stops["location_type"] == 1.0].copy()

print("Swiss GTFS all stops:", len(stops))
print("Swiss GTFS without parent_station:", len(gtfs_no_parent_ch))
print("Swiss GTFS with location_type == 1.0:", len(gtfs_loc1_ch))

Swiss GTFS all stops: 97067
Swiss GTFS without parent_station: 35267
Swiss GTFS with location_type == 1.0: 35267


## Result

The most important observation is that the two GTFS station-like subsets have exactly the same size:
- records without `parent_station`: 35,267
- records with `location_type == 1.0`: 35,267

At the same time, the Swiss NeTEx `StopPlace` subset has a very similar size: 35,324.

So in the Swiss case, the GTFS station-level subset and the NeTEx `StopPlace` subset are much closer in size than in the Austria case. This suggests that the station-level comparison may be more straightforward here.

## Compare data quality

In [35]:
def basic_profile(df, id_col, important_cols):
    result = {
        "rows": len(df),
        "unique_ids": df[id_col].nunique(dropna=True),
        "duplicate_ids": df[id_col].duplicated().sum()
    }
    for col in important_cols:
        result[f"missing_{col}"] = df[col].isna().sum()
        result[f"missing_{col}_pct"] = round(df[col].isna().mean() * 100, 2)
    return pd.Series(result)

candidate_profile_ch = pd.DataFrame({
    "GTFS_no_parent": basic_profile(
        gtfs_no_parent_ch,
        id_col="stop_id",
        important_cols=["stop_name", "stop_lat", "stop_lon"]
    ),
    "GTFS_loc_type_1": basic_profile(
        gtfs_loc1_ch,
        id_col="stop_id",
        important_cols=["stop_name", "stop_lat", "stop_lon"]
    ),
    "NeTEx_StopPlace": basic_profile(
        netex_stopplace_ch,
        id_col="stop_id",
        important_cols=["stop_name", "stop_lat", "stop_lon"]
    )
})

candidate_profile_ch

,GTFS_no_parent,GTFS_loc_type_1,NeTEx_StopPlace
rows,35267.0,35267.0,35324.0
unique_ids,35267.0,35267.0,35324.0
duplicate_ids,0.0,0.0,0.0
missing_stop_name,0.0,0.0,0.0
missing_stop_name_pct,0.0,0.0,0.0
missing_stop_lat,0.0,0.0,0.0
missing_stop_lat_pct,0.0,0.0,0.0
missing_stop_lon,0.0,0.0,0.0
missing_stop_lon_pct,0.0,0.0,0.0


## Output

This table shows that all three station-level subsets are very clean.

For both GTFS subsets and for the NeTEx `StopPlace` subset:
- the number of rows is equal to the number of unique IDs
- no duplicate IDs were found
- no missing stop names were found
- no missing coordinates were found

So in the Swiss case, the station-level subsets are not only close in size, but also very complete. This means they provide a strong basis for the first station-level matching test.

## First station matching test: cleaned ID overlap

In this step, I test whether the Swiss GTFS station subset and the Swiss NeTEx `StopPlace` subset already overlap after simple ID cleaning.

The goal is to check whether the two datasets use related station identifiers, even if the raw ID format is not exactly the same.

### Station ID Overlap: GTFS vs NeTEx

To compare stations between GTFS and NeTEx, the IDs first need to be cleaned 
and standardized, since the two formats store them differently.

The function `extract_station_core_id` does three things:
- If the value is missing, return nothing
- If the ID starts with "Parent" (as in GTFS), remove that prefix
- Otherwise extract the numeric part at the end of the ID

Once both ID columns are cleaned, the overlap is computed using an intersection (count how many IDs appear in both datasets).

In [36]:
def extract_station_core_id(x):
    if pd.isna(x):
        return None
    s = str(x).strip()

    if s.startswith("Parent"):
        return s.replace("Parent", "")

    m = re.search(r"(\d+)$", s)
    return m.group(1) if m else s

gtfs_station_ch = gtfs_loc1_ch.copy()
netex_station_ch = netex_stopplace_ch.copy()

gtfs_station_ch["station_id_core"] = gtfs_station_ch["stop_id"].apply(extract_station_core_id)
netex_station_ch["station_id_core"] = netex_station_ch["stop_id"].apply(extract_station_core_id)

gtfs_station_ids = set(gtfs_station_ch["station_id_core"].dropna())
netex_station_ids = set(netex_station_ch["station_id_core"].dropna())

exact_core_overlap = gtfs_station_ids.intersection(netex_station_ids)

id_overlap_summary_ch = pd.DataFrame({
    "metric": [
        "GTFS station subset IDs",
        "NeTEx StopPlace IDs",
        "Exact overlapping cleaned IDs"
    ],
    "value": [
        len(gtfs_station_ids),
        len(netex_station_ids),
        len(exact_core_overlap)
    ]
})

id_overlap_summary_ch

,metric,value
0,GTFS station subset IDs,35267
1,NeTEx StopPlace IDs,35324
2,Exact overlapping cleaned IDs,35257


## Output

This first matching test gives a very strong result.

After simple ID cleaning, the Swiss GTFS station subset and the Swiss NeTEx `StopPlace` subset show an overlap of 35,257 station IDs.

This means that, unlike in the Austria case, the Swiss station-level comparison already works very well at the identifier level after a simple normalization step.

So in simple words:
- the two datasets use different raw ID formats
- but they still seem to refer to almost the same station identifiers underneath
- this makes the Swiss station-level matching much more straightforward

## Merging matched stations into a single table

Now that we know 35,257 station IDs overlap, we merge the two datasets into 
one single table using the cleaned ID as the common key. 

The merge uses `how="inner"` which means only stations that exist in **both** 
GTFS and NeTEx are kept.

The `suffixes=("_gtfs", "_netex")` part handles columns that have the same name 
in both datasets (like `stop_name`) by adding a label at the end so we can tell 
them apart. For example `stop_name_gtfs` and `stop_name_netex`.

In [37]:
id_matches_ch = gtfs_station_ch.merge(
    netex_station_ch,
    on="station_id_core",
    how="inner",
    suffixes=("_gtfs", "_netex")
)

print("Matched rows after cleaned ID join:", len(id_matches_ch))

display(
    id_matches_ch[
        ["station_id_core", "stop_id_gtfs", "stop_name_gtfs", "stop_id_netex", "stop_name_netex"]
    ].head(10)
)

Matched rows after cleaned ID join: 35257


,station_id_core,stop_id_gtfs,stop_name_gtfs,stop_id_netex,stop_name_netex
0,1100008,Parent1100008,"Zell (Wiesental), Wilder Mann",ch:2:StopPlace:1100008,"Zell (Wiesental), Wilder Mann"
1,1100009,Parent1100009,"Zell (Wiesental), Grönland",ch:2:StopPlace:1100009,"Zell (Wiesental), Grönland"
2,1100010,Parent1100010,Atzenbach,ch:2:StopPlace:1100010,Atzenbach
3,1100011,Parent1100011,"Mambach, Brücke",ch:2:StopPlace:1100011,"Mambach, Brücke"
4,1100012,Parent1100012,"Mambach, Mühlschau",ch:2:StopPlace:1100012,"Mambach, Mühlschau"
5,1100013,Parent1100013,"Mambach, Silbersau",ch:2:StopPlace:1100013,"Mambach, Silbersau"
6,1100014,Parent1100014,"Fröhnd (Schwarzw), Wühre",ch:2:StopPlace:1100014,"Fröhnd (Schwarzw), Wühre"
7,1100015,Parent1100015,"Fröhnd (Schwarzw), Unterkastel",ch:2:StopPlace:1100015,"Fröhnd (Schwarzw), Unterkastel"
8,1100016,Parent1100016,Wembach (Baden),ch:2:StopPlace:1100016,Wembach (Baden)
9,1100017,Parent1100017,"Schönau (Schw), Brand",ch:2:StopPlace:1100017,"Schönau (Schw), Brand"


## Output

This output confirms the result from the previous step.

After joining the Swiss GTFS station subset and the Swiss NeTEx `StopPlace` subset on the cleaned station ID, I obtain 35,257 matched rows.

The example rows show that the matched records are highly plausible:
- the cleaned station IDs correspond to each other
- the GTFS and NeTEx stop names are the same in the shown examples

So in simple words:
- the ID-based matching is not only large in size
- it also looks correct in practice
- this means that the Swiss station-level comparison can rely strongly on cleaned IDs

## Summarize the station-level ID matching result

In this step, I summarize the result of the cleaned ID matching.

The goal is to see:
- how large the match is compared with the full GTFS and NeTEx station subsets
- how many records remain unmatched on each side

This helps me understand how complete the station-level overlap is in the Swiss case.

In [38]:
station_match_summary_ch = pd.DataFrame({
    "metric": [
        "GTFS station subset",
        "NeTEx StopPlace subset",
        "matched rows after cleaned ID join",
        "match rate vs GTFS subset (%)",
        "match rate vs NeTEx subset (%)"
    ],
    "value": [
        len(gtfs_station_ch),
        len(netex_station_ch),
        len(id_matches_ch),
        round(len(id_matches_ch) / len(gtfs_station_ch) * 100, 2),
        round(len(id_matches_ch) / len(netex_station_ch) * 100, 2)
    ]
})

station_match_summary_ch

,metric,value
0,GTFS station subset,35267.00
1,NeTEx StopPlace subset,35324.00
2,matched rows after cleaned ID join,35257.00
3,match rate vs GTFS subset (%),99.97
4,match rate vs NeTEx subset (%),99.81


In [39]:
matched_core_ids = set(id_matches_ch["station_id_core"])

gtfs_unmatched_ch = gtfs_station_ch[~gtfs_station_ch["station_id_core"].isin(matched_core_ids)].copy()
netex_unmatched_ch = netex_station_ch[~netex_station_ch["station_id_core"].isin(matched_core_ids)].copy()

print("Unmatched GTFS station rows:", len(gtfs_unmatched_ch))
print("Unmatched NeTEx StopPlace rows:", len(netex_unmatched_ch))

display(gtfs_unmatched_ch[["station_id_core", "stop_id", "stop_name"]].head(10))
display(netex_unmatched_ch[["station_id_core", "stop_id", "stop_name"]].head(10))

Unmatched GTFS station rows: 10
Unmatched NeTEx StopPlace rows: 67


,station_id_core,stop_id,stop_name
74896,8509442,Parent8509442,"Meggen, Fischerdörfli"
74937,8509551,Parent8509551,"St-Léonard, gare"
75223,8510023,Parent8510023,"Meggen, Schiffstation"
75459,8510499,Parent8510499,"Meggen, Meggenhorn"
75460,8510500,Parent8510500,"Meggen, Schwerzihöhe"
75461,8510501,Parent8510501,"Meggen, Stampfiweg"
75550,8510639,Parent8510639,"La Corbatière, parking"
75552,8510641,Parent8510641,"La Sagne-Eglise, halte"
75556,8510645,Parent8510645,"Petit-Martel-Est, halte"
75669,8510891,Parent8510891,"Meggen, Altstadstrasse"


,station_id_core,stop_id,stop_name
2,10,ch:2:StopPlace:10,"Petit-Martel-Est, halte"
3578,1104633,ch:2:StopPlace:1104633,"Waldshut, Im unteren Tal"
7825,132,ch:2:StopPlace:132,Bahn-2000-Strecke
7829,133,ch:2:StopPlace:133,Centovalli
7830,134,ch:2:StopPlace:134,Furka-Basistunnel
7831,135,ch:2:StopPlace:135,Lötschberg-Basistunnel
7832,136,ch:2:StopPlace:136,Lötschberg-Bergstrecke
7833,137,ch:2:StopPlace:137,Rotsee
7834,138,ch:2:StopPlace:138,Vereinatunnel
7835,139,ch:2:StopPlace:139,Seedamm


## Observation

Almost the entire station-level subset overlaps:
- 35,257 matched rows
- match rate of 99.97% from the GTFS side
- match rate of 99.81% from the NeTEx side

Only a very small number of records remain unmatched:
- 10 GTFS station records
- 67 NeTEx `StopPlace` records

So in the Swiss case, the station-level comparison works extremely well already with simple ID cleaning. This is much more direct than in the Austria case, where station matching needed normalized names and coordinate support.

## Conclusion for the station-level comparison

The Swiss station-level comparison works very well after simple ID cleaning. Almost the entire GTFS station subset overlaps with the NeTEx `StopPlace` subset, and only a very small number of records remain unmatched.

This means that, for Switzerland, the station-level mapping is much more direct than in the Austria case. I now move to the line level.

## 3.2 Line-level comparison

In [40]:
display(routes[["route_id", "route_short_name", "route_long_name", "route_type"]].head(10))
display(netex_lines_ch[["route_id", "route_name", "short_name", "public_code"]].head(10))

,route_id,route_short_name,route_long_name,route_type
0,91-10-A-j26-1,S10,NaN,109
1,91-10-B-j26-1,S10,NaN,109
2,91-10-C-j26-1,S10,NaN,109
3,91-10-D-j26-1,10,NaN,117
4,91-10-E-j26-1,10,NaN,900
5,91-10-j26-1,10,NaN,900
6,91-11-A-j26-1,S11,NaN,109
7,91-11-C-j26-1,11,NaN,900
8,91-11-E-j26-1,SN11,NaN,109
9,91-11-G-j26-1,R11,NaN,106


,route_id,route_name,short_name,public_code
0,ch:2:Line:06____.B.7,7,7,7
1,ch:2:Line:06____.RB.27,27,27,27
2,ch:2:Line:06____.RE.RE7,RE7,7,RE7
3,ch:2:Line:101.B.23,23,23,23
4,ch:2:Line:101.FUN.23,23,23,23
5,ch:2:Line:103.FUN.22,22,22,22
6,ch:2:Line:104.CC.7,7,7,7
7,ch:2:Line:105.FUN.2840,2840,284,2840
8,ch:2:Line:107.ASC.ASC,ASC,4V,ASC
9,ch:2:Line:107.FUN.FUN,FUN,84,FUN


## Observation 

The preview shows that both datasets contain usable line-level information, but the most relevant fields are not exactly the same.

In the Swiss GTFS table, the most visible public label seems to be `route_short_name`, while `route_long_name` is often not filled in the shown examples.

In the Swiss NeTEx table, line-related information appears in `route_name`, `short_name`, and `public_code`. In several examples, `route_name` and `public_code` look very similar or identical, while `short_name` can differ.

So at this stage, the line-level comparison should focus first on the fields that seem closest to passenger-facing line labels, especially:
- GTFS `route_short_name`
- NeTEx `public_code`
- and, if needed, NeTEx `route_name` as additional support

In [41]:
line_profile_ch = pd.DataFrame({
    "GTFS_routes": {
        "rows": len(routes),
        "unique_route_id": routes["route_id"].nunique(),
        "missing_route_short_name": routes["route_short_name"].isna().sum(),
        "missing_route_long_name": routes["route_long_name"].isna().sum(),
        "unique_route_short_name": routes["route_short_name"].nunique(dropna=True),
        "unique_route_long_name": routes["route_long_name"].nunique(dropna=True),
    },
    "NeTEx_lines": {
        "rows": len(netex_lines_ch),
        "unique_route_id": netex_lines_ch["route_id"].nunique(),
        "missing_route_name": netex_lines_ch["route_name"].isna().sum(),
        "missing_short_name": netex_lines_ch["short_name"].isna().sum(),
        "missing_public_code": netex_lines_ch["public_code"].isna().sum(),
        "unique_route_name": netex_lines_ch["route_name"].nunique(dropna=True),
        "unique_short_name": netex_lines_ch["short_name"].nunique(dropna=True),
        "unique_public_code": netex_lines_ch["public_code"].nunique(dropna=True),
    }
})

line_profile_ch

,GTFS_routes,NeTEx_lines
rows,4883.0,4014.0
unique_route_id,4883.0,4014.0
missing_route_short_name,0.0,NaN
missing_route_long_name,4883.0,NaN
unique_route_short_name,1792.0,NaN
unique_route_long_name,0.0,NaN
missing_route_name,NaN,0.0
missing_short_name,NaN,0.0
missing_public_code,NaN,0.0
unique_route_name,NaN,1792.0


## Result

Both datasets are complete at the line level: no missing values in the key 
name fields. However, GTFS has 4,883 route records and NeTEx has 4,014 line 
records, while both only have **1,792 unique public line labels**. This means 
many records share the same visible line name. In GTFS, `route_long_name` is 
entirely empty for Switzerland, so `route_short_name` is the only usable name 
field. In NeTEx, all three name fields are fully available.

In [42]:
print("Most frequent GTFS route_short_name values:")
display(routes["route_short_name"].value_counts().head(15))

print("Most frequent NeTEx public_code values:")
display(netex_lines_ch["public_code"].value_counts().head(15))

print("Most frequent NeTEx route_name values:")
display(netex_lines_ch["route_name"].value_counts().head(15))

Most frequent GTFS route_short_name values:


route_short_name
EXT    75
EV1    63
PB     54
EV2    52
EV     52
SL     36
GB     34
3      34
1      33
2      31
6      30
4      29
S      27
5      27
7      26
Name: count, dtype: int64

Most frequent NeTEx public_code values:


public_code
PB    44
EV    30
6     28
3     27
1     26
7     26
SL    26
2     26
4     24
5     23
8     23
12    21
10    19
9     19
11    18
Name: count, dtype: int64

Most frequent NeTEx route_name values:


route_name
PB    44
EV    30
6     28
3     27
1     26
7     26
SL    26
2     26
4     24
5     23
8     23
12    21
10    19
9     19
11    18
Name: count, dtype: int64

## Output

The most frequent labels like `PB`, `EV`, `SL` appear in both GTFS and NeTEx, 
which confirms that the two datasets cover the same lines. The counts differ 
slightly. For example `PB` appears 54 times in GTFS but 44 times in NeTEx, 
because each format splits routes into records differently internally. 
Importantly, `public_code` and `route_name` in NeTEx are identical, meaning 
they carry the same information.

This is why comparing by unique public line labels is the better approach:

| | GTFS | NeTEx | Match |
|---|---|---|---|
| Raw route ID | `26-PB-j26-1` | `ch:1:Line:PB:SBB` | No |
| Public label | `PB` | `PB` | Yes |

For example, the label `PB` exists in both datasets and refers to the same 
real-world line but its raw route ID looks like `26-PB-j26-1` in GTFS and 
`ch:1:Line:PB:SBB` in NeTEx. These two IDs would never match in a direct 
comparison, even though they represent the exact same line. The public label 
`PB` is the only thing both formats agree on.

## Compare the unique public line labels

In this step, I compare the unique visible line labels between GTFS and NeTEx.

I do not compare raw route IDs, because the earlier results showed that many route records share the same public label.

So here I focus on:
- GTFS `route_short_name`
- NeTEx `public_code`
- and as a support check, NeTEx `route_name`

To make the comparison more robust, I apply a simple normalization to the labels before comparing them.

### Line label overlap: GTFS vs NeTEx

- Normalize labels: uppercase and remove spaces so small differences don't cause mismatches
- Extract unique labels from GTFS `route_short_name`, NeTEx `public_code` and `route_name`
- Count how many labels overlap between GTFS and NeTEx
- Summarize the result in `line_overlap_summary_ch`

In [43]:
def normalize_line_label(x):
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    s = re.sub(r"\s+", "", s)   # remove spaces
    return s

routes["line_label_norm"] = routes["route_short_name"].apply(normalize_line_label)
netex_lines_ch["public_code_norm"] = netex_lines_ch["public_code"].apply(normalize_line_label)
netex_lines_ch["route_name_norm"] = netex_lines_ch["route_name"].apply(normalize_line_label)

gtfs_line_set = set(routes["line_label_norm"].dropna())
netex_public_code_set = set(netex_lines_ch["public_code_norm"].dropna())
netex_route_name_set = set(netex_lines_ch["route_name_norm"].dropna())

overlap_public_code = gtfs_line_set.intersection(netex_public_code_set)
overlap_route_name = gtfs_line_set.intersection(netex_route_name_set)

line_overlap_summary_ch = pd.DataFrame({
    "comparison": [
        "GTFS route_short_name vs NeTEx public_code",
        "GTFS route_short_name vs NeTEx route_name"
    ],
    "gtfs_unique_labels": [
        len(gtfs_line_set),
        len(gtfs_line_set)
    ],
    "netex_unique_labels": [
        len(netex_public_code_set),
        len(netex_route_name_set)
    ],
    "exact_overlap_labels": [
        len(overlap_public_code),
        len(overlap_route_name)
    ],
    "overlap_vs_gtfs_pct": [
        round(len(overlap_public_code) / len(gtfs_line_set) * 100, 2),
        round(len(overlap_route_name) / len(gtfs_line_set) * 100, 2)
    ],
    "overlap_vs_netex_pct": [
        round(len(overlap_public_code) / len(netex_public_code_set) * 100, 2),
        round(len(overlap_route_name) / len(netex_route_name_set) * 100, 2)
    ]
})

line_overlap_summary_ch

,comparison,gtfs_unique_labels,netex_unique_labels,exact_overlap_labels,overlap_vs_gtfs_pct,overlap_vs_netex_pct
0,GTFS route_short_name vs NeTEx public_code,1789,1789,1789,100.0,100.0
1,GTFS route_short_name vs NeTEx route_name,1789,1789,1789,100.0,100.0


## Output

This table shows the overlap of the unique visible line labels between the Swiss GTFS and Swiss NeTEx datasets after simple normalization.

The result is very strong:
- GTFS `route_short_name` and NeTEx `public_code` have exactly the same 1,789 unique normalized labels
- GTFS `route_short_name` and NeTEx `route_name` also have exactly the same 1,789 unique normalized labels

This means that, at the level of unique public line labels, the Swiss GTFS and Swiss NeTEx data are fully aligned in this extraction.

A small additional observation is that the number of unique labels is now 1,789 instead of the slightly higher raw count seen before. This happens because the normalization step merges labels that differ only in formatting, such as spaces or letter case.

In [44]:
matched_line_labels_ch = pd.DataFrame({
    "matched_label": sorted(list(overlap_public_code))[:50]
})

gtfs_unmatched_labels_ch = sorted(list(gtfs_line_set - netex_public_code_set))[:30]
netex_unmatched_labels_ch = sorted(list(netex_public_code_set - gtfs_line_set))[:30]

print("Example matched labels:")
display(matched_line_labels_ch)

print("Example GTFS labels not found in NeTEx public_code:")
display(pd.DataFrame({"gtfs_only_label": gtfs_unmatched_labels_ch}))

print("Example NeTEx public_code labels not found in GTFS:")
display(pd.DataFrame({"netex_only_label": netex_unmatched_labels_ch}))

Example matched labels:


,matched_label
0,001A
1,001B
2,001G
3,01
4,02
5,03
6,04
7,041F
8,041G
9,061


Example GTFS labels not found in NeTEx public_code:


,gtfs_only_label


Example NeTEx public_code labels not found in GTFS:


,netex_only_label


## Output

The preview of matched labels shows many plausible public line labels, such as `001A`, `01`, `02`, `10`, `100`, and `105`. At the same time, the tables for GTFS-only and NeTEx-only labels are empty.

So in simple words:
- the matched labels are not only high in number
- they also look realistic in practice
- and there are no unmatched labels left after normalization

This confirms that the Swiss GTFS and Swiss NeTEx line-level comparison works extremely well at the level of unique public line labels.

## Comparing line label counts between GTFS and NeTEx

For each unique line label, I count how many times it appears in GTFS and in 
NeTEx separately. Then I merge the two counts into one table using an outer join (labels that only exist in one dataset are also kept, with a count of 0 
for the other). Finally I compute the difference between the two counts to see 
where GTFS and NeTEx disagree the most.

In [45]:
gtfs_label_counts_ch = (
    routes["line_label_norm"]
    .value_counts()
    .rename_axis("line_label_norm")
    .reset_index(name="gtfs_count")
)

netex_label_counts_ch = (
    netex_lines_ch["public_code_norm"]
    .value_counts()
    .rename_axis("line_label_norm")
    .reset_index(name="netex_count")
)

label_count_compare_ch = gtfs_label_counts_ch.merge(
    netex_label_counts_ch,
    on="line_label_norm",
    how="outer"
).fillna(0)

label_count_compare_ch["gtfs_count"] = label_count_compare_ch["gtfs_count"].astype(int)
label_count_compare_ch["netex_count"] = label_count_compare_ch["netex_count"].astype(int)
label_count_compare_ch["count_diff"] = label_count_compare_ch["gtfs_count"] - label_count_compare_ch["netex_count"]
label_count_compare_ch["abs_diff"] = label_count_compare_ch["count_diff"].abs()

print("Labels with exactly the same count:", (label_count_compare_ch["count_diff"] == 0).sum())
print("Labels with different counts:", (label_count_compare_ch["count_diff"] != 0).sum())

label_count_compare_ch.sort_values("abs_diff", ascending=False).head(20)

Labels with exactly the same count: 1453
Labels with different counts: 336


,line_label_norm,gtfs_count,netex_count,count_diff,abs_diff
1241,EXT,75,10,65,65
1222,EV1,67,18,49,49
1227,EV2,52,10,42,42
1221,EV,52,30,22,22
1228,EV3,26,5,21,21
1685,S,27,7,20,20
1645,RE,22,5,17,17
1246,GB,34,18,16,16
1230,EV4,19,4,15,15
1254,IC,14,1,13,13


## Observation on line-label frequencies

This result shows that the Swiss GTFS and Swiss NeTEx datasets use the same set of public line labels, but not always with the same number of technical route records behind each label.

For many labels, the counts are identical. However, for some labels such as `EXT`, `EV1`, or `EV2`, GTFS contains noticeably more route records than NeTEx.

So in simple words:
- the visible line labels match very well
- but the internal route structure is not always organised in exactly the same way

This means that the Switzerland line-level comparison is very strong at the public label level, even though the technical route decomposition can still differ between the two standards.

## Conclusion for line-level comparison

- **Stations:** almost all IDs match after simple cleaning
- **Lines:** both datasets have exactly the same 1,789 unique public line labels after normalization

The only differences are technical. Each format groups routes into records 
differently, but they refer to the same real-world lines and stations.

## 3.3 Calendar-level comparison

## Note on the Switzerland calendar comparison

The Switzerland NeTEx SERVICECALENDAR file provides one overall validity 
window: 2025-12-14 to 2026-12-12. This matches the GTFS calendar range exactly.

A deeper calendar comparison at the level of individual service patterns and 
active dates was not carried out for Switzerland. As shown in the Austrian 
analysis, GTFS and NeTEx store calendar logic in fundamentally different 
structures, and aligning them requires significant processing work. Given that 
this analysis covers multiple countries, the detailed calendar methodology 
developed for Austria is considered the reference approach and is not repeated 
for each country individually.